In [ ]:
!pip install filterpy

In [ ]:
import os
import json
import cv2
import matplotlib.pyplot as plt
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

# ĐỌC DỮ LIỆU

In [ ]:
import os
import random

# 1. Cấu hình đường dẫn
img_root = "/kaggle/input/datasets/cinminhcit/walkingawareness-wad-all-size/resized_images_only"

# 2. TỰ ĐỘNG lấy danh sách folder
# Lấy tất cả folder trong đường dẫn img_root
all_folders = [f for f in os.listdir(img_root) if os.path.isdir(os.path.join(img_root, f))]

# Sắp xếp để thứ tự chạy nhất quán (hoặc random nếu muốn test ngẫu nhiên)
all_folders.sort() 
sequences = all_folders

print(f"Đã tìm thấy tổng cộng: {len(all_folders)} sequences.")
print("5 sequences đầu tiên:", sequences[:5])

In [ ]:
box_file = "/kaggle/input/datasets/cinminhcit/walkingawareness-wad-all-size/all_bboxes.jsonl"
all_boxes = []
with open(box_file, "r") as f:
    for line in f:
        obj = json.loads(line)
        all_boxes.append(obj)

In [ ]:
# Hàm gom nhóm toàn bộ bboxes theo folder id để phục
# vụ cho hàm load_sequence
from collections import defaultdict

boxes_by_folder = defaultdict(list)
for box in all_boxes:
    key = box['folder_id']
    boxes_by_folder[key].append(box)    

In [ ]:
from collections import defaultdict

def load_sequence(seq_name):
    seq_path = os.path.join(img_root, seq_name)
    
    # Load ảnh
    frames = sorted(
        [f for f in os.listdir(seq_path) if f.endswith(".jpg")],
        key=lambda x: int(x.split(".")[0])
    )

    assert len(frames) == 9, f"{seq_name} không có 9 frame!"

    images = [os.path.join(seq_path, f) for f in frames]

    # Lọc bbox của sequence đó (sử dụng boxes_by_folder)
    seq_boxes = boxes_by_folder.get(seq_name, [])
    
    # Gom theo frame_id
    boxes_by_frame = defaultdict(list)
    for b in seq_boxes:
        boxes_by_frame[b["frame_id"]].append(b)

    # Tạo list detection theo thứ tự frame
    detections = []
    for i in range(9):
        detections.append(boxes_by_frame.get(i, []))

    return images, detections


# Hàm bổ trợ

In [ ]:
def convert_bbox(bbox): 
    '''
    Chuyển từ [x1, y1, x2, y2] về  [x, y, s, r]
    '''
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]
    x = bbox[0] + w/2.
    y = bbox[1] + h/2.
    s = w * h
    r = w / float(h) if h > 0 else 1.0
    return np.array([[x], [y], [s], [r]]).reshape((4, 1))

def get_box_coords(state):
    """
    Chuyển từ để tính IOU
    """
    x, y, s, r = state
    w = np.sqrt(s * r)
    h = s / w
    x1 = x - w / 2.
    y1 = y - h / 2.
    x2 = x + w / 2.
    y2 = y + h / 2.
    return [x1, y1, x2, y2]

def calculate_iou(box1, box2):
    """
    box1, box2: [x1, y1, x2, y2]
    """
    xx1 = max(box1[0], box2[0])
    yy1 = max(box1[1], box2[1])
    xx2 = min(box1[2], box2[2])
    yy2 = min(box1[3], box2[3])

    w = max(0, xx2 - xx1)
    h = max(0, yy2 - yy1)
    inter = w * h
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    
    return inter / union if union > 0 else 0

def compute_color_hist(image, bbox):
    """
    Tính đặc trưng màu sắc (HSV) của một bounding box.
    """
    h_img, w_img = image.shape[:2]
    x1, y1, x2, y2 = map(int, bbox)
    
    # Clip coordinates để không lỗi biên
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(w_img, x2); y2 = min(h_img, y2)
    
    crop = image[y1:y2, x1:x2]
    if crop.size == 0: return None

    # Chuyển sang HSV
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    # Chỉ lấy Hue và Saturation
    hist = cv2.calcHist([hsv], [0, 1], None, [18, 20], [0, 180, 0, 256])
    
    cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
    return hist

def get_motion_info(trk, img_w, img_h):
    """Tính toán vận tốc, hướng di chuyển và chuẩn hoá với đường chéo khung hình"""
    vx = trk.kf.x[4].item()
    vy = trk.kf.x[5].item()
    
    # Tính tốc độ trên pixel
    raw_speed = np.sqrt(vx**2 + vy**2)
    angle = np.degrees(np.arctan2(vy, vx))        
    
    # Tính đường chéo của khung hình
    diagonal = np.sqrt(img_w**2 + img_h**2)
    
    # Chuẩn hoá tốc độ và vận tốc với đường chéo
    norm_speed = raw_speed / diagonal
    norm_vx = vx / diagonal
    norm_vy = vy / diagonal
    speed_percent = round(norm_speed * 100, 2)
    
    # Trả về cả giá trị chuẩn hoá (cho JSON) và giá trị gốc (cho Visualization)
    return speed_percent, norm_vx, norm_vy, angle, raw_speed, vx, vy

In [ ]:
def create_json_entry(raw_det, track_id, speed_pct, clock_position, azimuth_deg, vx=0.0, vy=0.0):
    """
    Tạo một dictionary mới chứa dữ liệu gốc cộng thêm kết quả tracking.
    """
    entry = raw_det.copy() 
    entry["track_id"] = int(track_id)
    entry["speed_percent"] = round(float(speed_pct), 2)
    entry["clock_position"] = clock_position
    entry["azimuth_deg"] = azimuth_deg
    entry["vx"] = round(float(vx), 4) 
    entry["vy"] = round(float(vy), 4)
    
    return entry

In [ ]:
import math

# Xác định hướng đồng hồ
# -------------------------------------------------
# TÍNH f MỘT LẦN DUY NHẤT
# -------------------------------------------------
def compute_focal_length(image_width, fov_deg=90):
    f = (image_width / 2) / math.tan(math.radians(fov_deg / 2))
    return f

# -------------------------------------------------
# TÍNH AZIMUTH CHO MỖI OBJECT
# -------------------------------------------------
def compute_azimuth_from_bbox(bbox, image_width, f):
    x1, y1, x2, y2 = bbox
    cx = (x1 + x2) / 2.0
    dx = cx - image_width / 2
    azimuth_deg = math.degrees(math.atan2(dx, f))
    return azimuth_deg

# -------------------------------------------------
# AZIMUTH → CLOCK
# -------------------------------------------------
def azimuth_to_clock_10_to_2(azimuth_deg):
    if azimuth_deg <= -30:
        return "10 o'clock"
    elif azimuth_deg <= -10:
        return "11 o'clock"
    elif azimuth_deg <= 10:
        return "12 o'clock"
    elif azimuth_deg <= 30:
        return "1 o'clock"
    else:
        return "2 o'clock"

# -------------------------------------------------
# HÀM CHÍNH CHO MỖI OBJECT 
# -------------------------------------------------
def compute_clock_info_10_to_2(bbox, image_width, f):
    azimuth_deg = compute_azimuth_from_bbox(bbox, image_width, f)
    clock_str = azimuth_to_clock_10_to_2(azimuth_deg)

    return {
        "azimuth_deg": round(azimuth_deg, 2),
        "clock_position": clock_str,
    }

In [ ]:
def print_tabular_report(frame_idx, json_results):
    print(f"\n>>> BÁO CÁO FRAME {frame_idx}")
    
    header = (
        f"{'ID':<4} | "
        f"{'Label':<10} | "
        f"{'Speed (%)':<9} | " 
        f"{'Clock':<12} | "
    )
    
    print(header)
    print("-" * len(header))

    for item in json_results:
        track_id = item.get("track_id", -1)
        label = item.get("label", "unknown")
        speed = item.get("speed_percent", 0.0)
        clock = item.get("clock_position", "-")
        print(
            f"{track_id:<4} | "
            f"{label:<10} | "
            f"{speed:<9.2f} | " 
            f"{clock:<12} | "
        )

    print("=" * len(header))

# Kalman + Association

In [ ]:
import numpy as np
from filterpy.kalman import KalmanFilter

class Tracker:
    def __init__(self, track_id, initial_box, label, initial_image): # <--- THÊM initial_image
        self.id = track_id
        self.label = label
        self.time_since_update = 0
        self.hits = 0 
        # self.velocity_smooth = np.array([0.0, 0.0]) 
        self.hist = None

        box = convert_bbox(initial_box) 
        # Dùng .item() chuẩn hơn float() cho numpy array
        self.last_pos = np.array([box[0].item(), box[1].item()]) 

        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        dt = 1 
        
        self.kf.F = np.array([
            [1, 0, 0, 0, dt, 0, 0], 
            [0, 1, 0, 0, 0, dt, 0], 
            [0, 0, 1, 0, 0, 0, dt],
            [0, 0, 0, 1, 0, 0, 0],  
            [0, 0, 0, 0, 1, 0, 0],  
            [0, 0, 0, 0, 0, 1, 0], 
            [0, 0, 0, 0, 0, 0, 1]
        ])
        self.kf.H = np.array([
            [1, 0, 0, 0, 0, 0, 0], 
            [0, 1, 0, 0, 0, 0, 0], 
            [0, 0, 1, 0, 0, 0, 0], 
            [0, 0, 0, 1, 0, 0, 0]
        ])
        
        self.kf.R[2:, 2:] *= 10.  
        self.kf.P[4:, 4:] *= 1000.
        self.kf.P *= 10.
        self.kf.Q[4:, 4:] *= 0.01
        self.kf.x[:4] = convert_bbox(initial_box)

        # Tính toán histogram ban đầu
        # Lưu ý: Hàm compute_color_hist phải được định nghĩa ở ngoài class này
        self.hist = compute_color_hist(initial_image, initial_box)


    def apply_camera_motion(self, M):
        """
        Dịch chuyển vị trí dự đoán theo chuyển động của Camera
        """
        cx = self.kf.x[0].item()
        cy = self.kf.x[1].item()
        pos = np.array([cx, cy, 1.0]) 
        new_pos = M @ pos 
        self.kf.x[0] = new_pos[0]
        self.kf.x[1] = new_pos[1]
        
        last_pos_vec = np.array([self.last_pos[0], self.last_pos[1], 1.0])
        new_last_pos = M @ last_pos_vec
        
        self.last_pos[0] = new_last_pos[0]
        self.last_pos[1] = new_last_pos[1]

    def predict(self):
        if (self.kf.x[6].item() + self.kf.x[2].item()) <= 0: 
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.time_since_update += 1

    def update(self, bbox, raw_det=None, image=None): # <--- THÊM image=None
        self.time_since_update = 0
        self.hits += 1      
        self.kf.update(convert_bbox(bbox)) 

        # --- HIỂN THỊ VẬN TỐC & HƯỚNG ---
        raw_vx = self.kf.x[4].item()
        raw_vy = self.kf.x[5].item()
        
        self.last_pos = np.array([self.kf.x[0].item(), self.kf.x[1].item()])
        
        if raw_det is not None:
            self.last_raw_det = raw_det
            
        if image is not None:
            new_hist = compute_color_hist(image, bbox) 
            
            if self.hist is None:
                self.hist = new_hist
            elif new_hist is not None:
                # Cập nhật mềm: giữ 70% cũ, nạp 30% mới
                self.hist = 0.7 * self.hist + 0.3 * new_hist
                # QUAN TRỌNG: Chuẩn hóa lại về dải [0, 1]
                cv2.normalize(self.hist, self.hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)


In [ ]:
def mahalanobis_distance(trk, det_cx, det_cy):
    z = np.array([det_cx, det_cy])
    x = trk.kf.x[:2].flatten()
    P = trk.kf.P[:2, :2]
    S = P + trk.kf.R[:2, :2]  # innovation covariance
    diff = z - x
    return np.sqrt(diff.T @ np.linalg.inv(S) @ diff)

In [ ]:
def associate_object(detections, trackers, current_image):
    if len(trackers) == 0:
        return np.empty((0, 2), dtype=int), list(range(len(detections))), []

    # =========================================================
    # VÒNG 1: HIGH CONFIDENCE (Dựa trên IoU - Strict)
    # =========================================================
    iou_matrix = np.zeros((len(trackers), len(detections)))
    for t, trk in enumerate(trackers):
        trk_rect = get_box_coords(trk.kf.x.flatten()[:4])
        for d, det in enumerate(detections):
            if trk.label != det['label']:
                iou_matrix[t, d] = 1.0 
                continue
            iou = calculate_iou(trk_rect, det['boxs'])
            iou_matrix[t, d] = 1.0 - iou

    row_ind, col_ind = linear_sum_assignment(iou_matrix)

    matches_1 = []
    matched_trk_idx = set()
    matched_det_idx = set()
    iou_threshold = 0.8  # IoU phải > 0.2

    for r, c in zip(row_ind, col_ind):
        if iou_matrix[r, c] < iou_threshold:
            matches_1.append([r, c])
            matched_trk_idx.add(r)
            matched_det_idx.add(c)

    unmatched_trks = [t for t in range(len(trackers)) if t not in matched_trk_idx]
    unmatched_dets = [d for d in range(len(detections)) if d not in matched_det_idx]


    # =========================================================
    # VÒNG 2: RECOVERY (MÀU SẮC + KHOẢNG CÁCH XA)
    # =========================================================
    matches_2 = []

    if len(unmatched_trks) > 0 and len(unmatched_dets) > 0:
        cost_matrix_2 = np.zeros((len(unmatched_trks), len(unmatched_dets)))

        for i, t_idx in enumerate(unmatched_trks):
            trk = trackers[t_idx]
            t_box = get_box_coords(trk.kf.x.flatten()[:4])
            t_cx, t_cy = (t_box[0]+t_box[2])/2, (t_box[1]+t_box[3])/2
            
            for j, d_idx in enumerate(unmatched_dets):
                det = detections[d_idx]
                if trk.label != det['label']:
                    cost_matrix_2[i, j] = 1e9
                    continue

                # 1. Cost Màu
                det_hist = compute_color_hist(current_image, det['boxs'])
                d_color = 1.0
                if trk.hist is not None and det_hist is not None:
                    d_color = cv2.compareHist(trk.hist, det_hist, cv2.HISTCMP_BHATTACHARYYA)

                # 2. Cost Khoảng Cách
                d_box = det['boxs']
                d_cx, d_cy = (d_box[0]+d_box[2])/2, (d_box[1]+d_box[3])/2
                d_h = d_box[3] - d_box[1]
                dist_px = np.sqrt((t_cx - d_cx)**2 + (t_cy - d_cy)**2)
                norm_dist = dist_px / d_h if d_h > 0 else 100.0

                # 3. Cost Mahalanobis
                d_maha = mahalanobis_distance(trk, d_cx, d_cy)

                # GATING
                if d_color > 0.4 or norm_dist > 3.0 or d_maha > 3.0:
                    cost_matrix_2[i, j] = 1e5
                else:
                    cost_matrix_2[i, j] = 0.7 * d_color + 0.2 * (norm_dist / 3.0) + 0.1 * (d_maha / 3.0)

        r_ind_2, c_ind_2 = linear_sum_assignment(cost_matrix_2)
        
        for r, c in zip(r_ind_2, c_ind_2):
            if cost_matrix_2[r, c] < 0.5:
                matches_2.append([unmatched_trks[r], unmatched_dets[c]])

    # =========================================================
    # TỔNG HỢP
    # =========================================================
    final_matches = list(matches_1) + matches_2
    final_matches = np.array(final_matches) if len(final_matches) > 0 else np.empty((0, 2), dtype=int)
    
    if len(final_matches) > 0:
        matched_trk_set = set(final_matches[:, 0])
        matched_det_set = set(final_matches[:, 1])
    else:
        matched_trk_set = set()
        matched_det_set = set()

    final_unmatched_dets = [d for d in range(len(detections)) if d not in matched_det_set]
    final_unmatched_trks = [t for t in range(len(trackers)) if t not in matched_trk_set]

    return final_matches, final_unmatched_dets, final_unmatched_trks



In [ ]:
class CameraMotionEstimator:
    def __init__(self):
        self.detector = cv2.ORB_create(nfeatures=500)
        self.matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        self.prev_kp = None
        self.prev_des = None
        
    def estimate(self, curr_frame):
        curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
        curr_kp, curr_des = self.detector.detectAndCompute(curr_gray, None)

        if self.prev_kp is None or curr_des is None or len(curr_kp) < 20:
            self.prev_kp = curr_kp
            self.prev_des = curr_des
            return np.eye(3)

        matches = self.matcher.match(self.prev_des, curr_des)
        src_pts = []
        dst_pts = []
        for m in matches:
            src_pts.append(self.prev_kp[m.queryIdx].pt)
            dst_pts.append(curr_kp[m.trainIdx].pt)

        src_pts = np.float32(src_pts).reshape(-1, 1, 2)
        dst_pts = np.float32(dst_pts).reshape(-1, 1, 2)

        if len(matches) > 10:
            M, mask = cv2.estimateAffinePartial2D(src_pts, dst_pts, method=cv2.RANSAC)
            if M is None: 
                M_3x3 = np.eye(3)
            else:
                M_3x3 = np.eye(3)
                M_3x3[:2] = M
        else:
            M_3x3 = np.eye(3)

        self.prev_kp = curr_kp
        self.prev_des = curr_des
        return M_3x3

# Thêm xu hướng chuyển động

In [ ]:
def predict_collision_risk(item, image_width = 1280, image_height = 720, corridor_ratio = 0.4):
    """
    Xác định vật thể có hướng di chuyển cắt ngang 'Hành lang an toàn' của người dùng hay không.
    - corridor_ratio: Tỷ lệ bề ngang của hành lang an toàn (0.4 = chiếm 40% khu vực giữa ảnh).
    """
    # khởi tạo giá trị mặc định
    item['is_approaching'] = False
    item['will_collide'] = False
    # Bỏ qua nếu ko có dữ liệu tracking 
    if 'vx' not in item or 'vy' not in item:
        return item

    # Chuyển vx, vy sang lại hệ pixel bằng cách nhân với đường chéo ảnh
    diagonal = np.sqrt(image_width**2 + image_height**2)
    v_x = item['vx'] * diagonal
    v_y = item['vy'] * diagonal
    
    # Lọc 1: Vật thể có đang tiến đến camera ko: v_y > 0 
    if v_y <= 0:
        return item
        
    item['is_approaching'] = True
    # Lọc 2: Dự đoán quỹ đạo rơi vào hành lang
    # Xác định vị trí hiện tại của vật thể: Trung điểm cạnh dưới của bbox
    xmin, ymin, xmax, ymax = item['boxs']
    x_c = ((xmin + xmax) / 2.0) * image_width # chuyển về hệ pixel
    y_bottom = ymax * image_height # chuyển về hệ pixel
    
    # Nếu vật thể đã nằm dưới đáy ảnh, bỏ qua
    if y_bottom >= image_height:
        return item
    
    # Tính điểm rơi tương lai
    # Tính số frames (t) cần thiết để vật thể chạm đáy ảnh (y = image_height)
    t = (image_height - y_bottom) / v_y
    x_future = x_c + (v_x * t)
    # Xác định ranh giới an toàn của user
    # Ví dụ ảnh rộng 1280, corridor_ratio = 0.4 -> Hành lang rộng 512px nằm ở giữa ảnh
    corridor_width = image_width * corridor_ratio
    x_min_safe = (image_width - corridor_width) / 2.0
    x_max_safe = (image_width + corridor_width) / 2.0
    
    # So sánh với ranh giới an toàn
    if x_min_safe <= x_future and x_future <= x_max_safe:
        item['will_collide'] = True
    # (Optional) Lưu lại X_future để bạn dễ debug hoặc vẽ line dự đoán lên ảnh
    item['predicted_x_future'] = round(x_future, 2)
    return item

# Thêm vị trí tương đối

In [ ]:
def analyze_object_trajectory_and_distance(item, image_width = 1280, image_height = 720, corridor_ratio = 0.4):
    item['distance_zone'] = 'Unknown'
    item['distance_px'] = None
    
    # Tinh toan khoang cach tuong doi va chia vung
    xmin, ymin, xmax, ymax = item['boxs']
    y_bottom = ymax * image_height # chuẩn hóa lại sang hệ tọa độ pixel
    
    distance_px = image_height - y_bottom
    item['distance_px'] = round(distance_px, 2)

    # chia vung theo ty le
    if y_bottom < (image_height * 0.5):
        item['distance_zone'] = 'Far'
    elif y_bottom < (image_height * 0.75): # Có thể đổi -> 0.8
        item['distance_zone'] = 'Quite close'
    else:
        item['distance_zone'] = 'Close'
    return item

# Scale lên toàn bộ data

In [ ]:
# Đầu vào là 1 video gồm 9 frame
# Mỗi frame sẽ được đi qua model object detection
# Mỗi frame sẽ có n json. Mỗi json là 1 object được detect ở frame đó        

In [ ]:
# Lấy ra các video
vids = set()
for item in all_boxes:
    vids.add(item.get('folder_id'))

vids = list(vids)
vids.sort()

In [ ]:
len(vids)

In [ ]:
vids2 = vids[0:10]

In [ ]:
len(vids2)

# Vòng lặp chính

In [ ]:
# Ghi trực tiếp ra file thay vì đẩy vào tracked_results gây tốn RAM
output_file_path = "/kaggle/working/results_streaming.jsonl"

if os.path.exists(output_file_path):
    os.remove(output_file_path)

In [ ]:
# Chạy kalman filter trên cả tập data -> Dự đoán hướng ở frame cuối
# Cấu hình màu sắc
CONFIRMED_COLOR = (0, 255, 0) # Xanh lá
NEW_COLOR = (0, 255, 255)     # Vàng

# ============================================================
# 2. VÒNG LẶP TRACKING & VISUALIZE
# ============================================================
# tracked_results = [] # TẠO LIST MỚI ĐỂ LƯU TOÀN BỘ KẾT QUẢ TRACKING

for seq in vids2:
    print(f"\n{'='*60}\nProcessing Sequence: {seq}\n{'='*60}")
    
    image_paths, seq_detections = load_sequence(seq)
    if not image_paths: continue

    motion_estimator = CameraMotionEstimator()
    tracker_list = []
    track_id_count = 1
    focal  = None

    for k in range(len(image_paths)):
        img = cv2.imread(image_paths[k])
        if img is None: continue
        img_h, img_w = img.shape[:2]

        if focal is None:
            focal = compute_focal_length(img_w, fov_deg=90)

        raw_frame_dets = seq_detections[k]
        
        valid_dets = []
        small_dets = []

        min_area = img_w * img_h * 0.002 # Tính diện tích tối thiểu
        # MỚI: Tạo một mảng giữ chỗ trước để duy trì đúng thứ tự gốc của JSON
        frame_results = [None] * len(raw_frame_dets)
        # MỚI: Dùng enumerate để lấy vị trí gốc (idx)
        for idx, det in enumerate(raw_frame_dets):
            x1_n, y1_n, x2_n, y2_n = det['boxs']
            box_px = [x1_n*img_w, y1_n*img_h, x2_n*img_w, y2_n*img_h]
            
            det_obj = {
                "raw_det": det,
                "boxs": box_px,
                "label": det["label"],
                "orig_idx": idx   # <--- LƯU LẠI VỊ TRÍ GỐC
            }
        # Tính diện tích (width * height) bằng toạ độ pixel
            w = box_px[2] - box_px[0]
            h = box_px[3] - box_px[1]
        # Phân loại ngay tại đây
            if (w * h) >= min_area:
                valid_dets.append(det_obj)
            else:
                small_dets.append(det_obj)
        # -----------------------------------------------------------------
        # PHẦN 1: XỬ LÝ VẬT THỂ NHỎ (Chỉ lưu JSON, track_id = -1, speed = 0)
        # -----------------------------------------------------------------
        for det_data in small_dets:
            box_px = det_data['boxs']
            
            clock_info = compute_clock_info_10_to_2(box_px, img_w, focal)
            
            new_entry = create_json_entry(
                raw_det=det_data['raw_det'],
                track_id=-1,     # Đánh dấu là không được track
                speed_pct=0.0,
                clock_position=clock_info["clock_position"],
                azimuth_deg=clock_info["azimuth_deg"],
                vx=0.0,
                vy=0.0
            )
            # SỬA: Gán đúng vào vị trí cũ thay vì dùng append
            frame_results[det_data['orig_idx']] = new_entry
        # -----------------------------------------------------------------
        # PHẦN 2: TRACKING CÁC VẬT THỂ ĐỦ LỚN (valid_dets)
        # -----------------------------------------------------------------
        M = motion_estimator.estimate(img)
        for trk in tracker_list:
            trk.predict()
            trk.apply_camera_motion(M)

        # Truyền valid_dets vào hàm associate_object 
        matches, unmatched_dets, _ = associate_object(valid_dets, tracker_list, img)

        # --- A. CẬP NHẬT MATCHED TRACKERS ---
        for trk_idx, det_idx in matches:
            trk = tracker_list[trk_idx]
            det_data = valid_dets[det_idx] # Dùng valid_dets
            
            box_px = det_data['boxs']
            raw_det = det_data['raw_det'] # Lấy lại dict gốc
            
            trk.update(box_px, raw_det=raw_det, image=img)
            clock_info = compute_clock_info_10_to_2(box_px, img_w, focal)
            
            # CHÚ Ý: Đón biến norm_vx và norm_vy ở đây
            speed_pct, norm_vx, norm_vy, _, raw_speed, vx, vy = get_motion_info(trk, img_w, img_h)

            # --- GỌI HÀM TẠO JSON ENTRY VÀ LƯU VÀO MẢNG TẠM ---
            new_entry = create_json_entry(
                raw_det=raw_det,
                track_id=trk.id,
                speed_pct=speed_pct,
                clock_position=clock_info["clock_position"],
                azimuth_deg = clock_info["azimuth_deg"],
                vx = norm_vx, 
                vy = norm_vy
            )
            # SỬA: Gán đúng vào vị trí cũ thay vì dùng append
            frame_results[det_data['orig_idx']] = new_entry
            
            # VẼ LÊN ẢNH
            x1, y1, x2, y2 = [int(v) for v in box_px]
            color = CONFIRMED_COLOR if trk.hits > 1 else NEW_COLOR
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, f"ID:{trk.id} {clock_info['clock_position']}", (x1, y1-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
            
            if raw_speed > 2.0: # Vẽ mũi tên nếu có chuyển động
                cx, cy = (x1+x2)//2, (y1+y2)//2
                cv2.arrowedLine(img, (cx, cy), (int(cx+vx), int(cy+vy)), (0,0,255), 2)
        # --- B. TẠO NEW TRACKERS ---
        for det_idx in unmatched_dets:
            det_data = valid_dets[det_idx] # Dùng valid_dets
            
            box_px = det_data['boxs']
            raw_det = det_data['raw_det']
            
            new_trk = Tracker(track_id_count, box_px, det_data["label"], initial_image=img)
            tracker_list.append(new_trk)
            track_id_count += 1
            
            clock_info = compute_clock_info_10_to_2(box_px, img_w, focal)
            
            # --- GỌI HÀM TẠO JSON ENTRY CHO OBJECT MỚI (speed = 0.0) ---
            new_entry = create_json_entry(
                raw_det=raw_det,
                track_id=new_trk.id,
                speed_pct=0.0,
                clock_position=clock_info["clock_position"],
                azimuth_deg = clock_info["azimuth_deg"]
            )
            # SỬA: Gán đúng vào vị trí cũ thay vì dùng append
            frame_results[det_data['orig_idx']] = new_entry

            # VẼ LÊN ẢNH
            x1, y1, x2, y2 = [int(v) for v in box_px]
            cv2.rectangle(img, (x1, y1), (x2, y2), NEW_COLOR, 2)
            cv2.putText(img, f"NEW:{new_trk.id}", (x1, y1-10), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, NEW_COLOR, 1)  
        # MỚI: Thêm toàn bộ các object của frame này vào kết quả tổng
        # Ghi thẳng kết quả vào file
        with open(output_file_path, "a", encoding="utf-8") as f:
            for item in frame_results:
                if item is not None:
                    f.write(json.dumps(item, ensure_ascii=False) + "\n")
                    
        # Dọn dẹp tracker cũ
        tracker_list = [t for t in tracker_list if t.time_since_update <= 30]

print(f"\n>>> XỬ LÝ HOÀN TẤT!")

In [ ]:
import json
# Chỉ lấy ra các frame 8 và thêm thuộc tính vị trí tương đối, xu hướng chuyển động
video_and_frame_8 = {}

# for obj in tracked_results:
with open(output_file_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue # bỏ qua dòng rỗng

        obj = json.loads(line)
        if obj.get('frame_id') == 8:
            key = (obj.get('folder_id'), obj.get('frame_id'))
            if video_and_frame_8.get(key) is None:
                video_and_frame_8[key] = {"dets": [obj]}
            else:
                video_and_frame_8[key]["dets"].append(obj)

In [ ]:
# Tính kích thước của frame ảnh
for key, _ in video_and_frame_8.items():
    image_path = img_root + "/" + key[0] + "/8.jpg"
    img = cv2.imread(image_path)

    img_h, img_w = img.shape[:2]
    video_and_frame_8[key]['image_height'] = img_h
    video_and_frame_8[key]['image_width'] = img_w

In [ ]:
 # Thêm thuộc tính vị trí tương đối, xu hướng chuyển động
for key, item in video_and_frame_8.items():
    img_h = video_and_frame_8[key].get('image_height')
    img_w = video_and_frame_8[key].get('image_width')
    for i in range(len(item["dets"])):
        # xu hướng chuyển động
        item["dets"][i] = predict_collision_risk(item["dets"][i], image_width=img_w, image_height=img_h)
        item["dets"][i]['red_alert'] = bool(item["dets"][i].get('is_approaching') or item["dets"][i].get('will_collide'))
        # vị trí tương đối
        item["dets"][i] = analyze_object_trajectory_and_distance(item["dets"][i], image_width=img_w, image_height=img_h)


In [ ]:
# Tính trung distance và azimuth
for key, item in video_and_frame_8.items():
    valid_distances = []
    abs_azimuths = []
    for obj in item["dets"]:
        valid_distances.append(obj.get('distance_px'))
        abs_azimuths.append(abs(obj.get('azimuth_deg')))
    median_dist = np.median(valid_distances) if valid_distances else 0
    median_azimuth = np.median(abs_azimuths)
    video_and_frame_8[key]['median_dist'] = median_dist
    video_and_frame_8[key]['median_azimuth'] = median_azimuth

# Bổ sung vào cách lọc hiện tại

In [ ]:
import json
import math

# Ngưỡng tối thiểu để coi là một đối tượng nguy hiểm (từ 0 đến 1)
DANGER_THRESHOLD = 0.3

MIN_THRESHOLD = 5

scored_objects = []
selected_objects = {} # phục vụ cho vẽ ảnh
final_data = {}

for key, item in video_and_frame_8.items():
    all_scored_objects = []
    selected_objects[key] = []
    for obj in item['dets']:
        # --- 1. Tính điểm Collision (S_collision) ---
        # Kết hợp is_approaching và will_collide
        score_collision = 0
        if obj.get('is_approaching'): score_collision += 0.6
        if obj.get('will_collide'): score_collision += 0.4
        
        # --- 2. Tính điểm Direction (S_direction) ---
        # Ưu tiên vật thể sát hướng 12h (0 độ)
        azi = abs(obj.get("azimuth_deg", 180))
        # Chuẩn hóa: 0 độ -> 1.0 điểm, 45 độ trở lên -> 0 điểm
        # score_direction = max(0, 1 - (azi / 45)) 
        score_direction = max(0, math.cos(math.radians(azi)))
        
        # --- 3. Tính điểm Proximity (S_proximity) ---
        # Kết hợp distance_zone và distance_px
        zone_map = {'Quite close': 0.7, 'Close': 1.0, 'Far': 0.4}
        score_zone = zone_map.get(obj.get('distance_zone'), 0.1)
        
        # Distance_px: càng nhỏ càng gần (max là image_height)
        max_distance_px = item['image_height']
        dist_px = obj.get('distance_px', max_distance_px)
        score_dist_px = max(0, 1 - (dist_px / max_distance_px))
        
        score_proximity = (score_zone + score_dist_px) / 2
    
        # --- TỔNG HỢP SCORE ---
        final_score = (score_collision * 0.25) + (score_direction * 0.4) + (score_proximity * 0.35)

        # Gán điểm vào object gốc để dùng sau
        obj['danger_score'] = final_score
        
        # Tạo bản copy sạch sẽ
        clean_obj = {
            "folder_id": obj.get('folder_id'),
            "frame_id": obj.get('frame_id'),
            "label": obj.get("label"),
            "probs": obj.get("probs"),
            "track_id": obj.get("track_id"),
            "danger_score": final_score,
            "distance_zone": obj.get("distance_zone"),
            "distance_px": obj.get("distance_px"),
            "speed": obj.get("speed_percent"),
            "coming_to_user": obj.get("red_alert"),
            "relative_postion": obj.get("clock_position"),
            "boxs": obj.get("boxs")
        }
        
        all_scored_objects.append(clean_obj)
        
    # Sắp xếp tất cả theo điểm số từ cao xuống thấp   
    all_scored_objects.sort(key=lambda x: x['danger_score'], reverse=True)
    
    # 1: Lấy những obj nguy hiểm (vượt ngưỡng)
    scored_objects = [d for d in all_scored_objects if d['danger_score'] >= DANGER_THRESHOLD]

    # 2: Nếu chưa đủ số lượng tối thiểu, lấy thêm những obj đứng đầu danh sách còn lại
    if len(scored_objects) < MIN_THRESHOLD:
        # Lấy thêm cho đủ MIN_THRESHOLD nhưng không vượt quá tổng số lượng có sẵn
        scored_objects = all_scored_objects[:MIN_THRESHOLD]
    
    final_data[key] = scored_objects    

In [ ]:
# Kiểm tra số lượng được lọc ở mỗi vid
for key, item in final_data.items():
    print(f"Video : {key}")
    print(f"Số lượng object có trong frame 8: {len(video_and_frame_8.get(key)["dets"])}")
    print(f"Số lượng object được lọc trong frame 8: {len(item)}")
    print(f"-----------------------------------------------------")

# Visualize để kiểm tra

In [ ]:
# Vẽ các frame 8 ra
final_data

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def draw_object_properties(img, obj):
    """
    Vẽ BBox và ghi các thông tin quan trọng của object lên ảnh.
    """
    img_h, img_w = img.shape[:2]

    # 1. Trích xuất tọa độ BBox
    if 'boxs' in obj:
        xmin, ymin, xmax, ymax = obj['boxs']
        x1, y1 = int(xmin * img_w), int(ymin * img_h)
        x2, y2 = int(xmax * img_w), int(ymax * img_h)
    else:
        return img 

    # 2. Phân loại màu sắc theo danger_score
    score = obj.get('danger_score', 0)
    if score >= 0.5:
        color = (0, 0, 255)       # Đỏ (Nguy hiểm cao)
    elif score >= 0.3:
        color = (0, 165, 255)     # Cam (Cảnh báo)
    else:
        color = (0, 255, 0)       # Xanh lá (An toàn)

    # 3. Vẽ Bounding Box
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

    # 4. Chuẩn bị thông tin rút gọn
    coming_status = "Yes" if obj.get('coming_to_user') else "No"
    speed_val = obj.get('speed', 0)
    if speed_val is None: speed_val = 0
    
    properties = [
        f"ID:{obj.get('track_id', '?')} | Score:{score}",
        f"Pos:{obj.get('relative_postion', '-')} | Zone:{obj.get('distance_zone', '-')}",
        f"Speed:{round(speed_val, 1)}% | Coming:{coming_status}"
    ]

    # 5. Cấu hình tọa độ Text
    text_x = x1
    text_y = y1 - 10 
    
    # Nếu y1 quá sát mép trên ảnh, đẩy text xuống dưới BBox để không bị khuất
    if y1 < 60:
        text_y = y2 + 20

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.5
    thickness = 2
    line_height = 18 # Khoảng cách giữa các dòng chữ

    # 6. Viết chữ trực tiếp lên ảnh (Không dùng viền/nền)
    for i, text in enumerate(properties):
        current_y = text_y + i * line_height
        # Sử dụng biến 'color' (màu của BBox) làm màu chữ luôn
        cv2.putText(img, text, (text_x, current_y), font, font_scale, color, thickness)

    return img

def visualize_last_frames_from_final_data(final_data_dict, img_root_path):
    """
    Duyệt trực tiếp từ final_data (biến chứa list object của từng frame 8)
    và vẽ hình ảnh.
    """
    for key, objects_list in final_data_dict.items():
        folder_id, frame_id = key
        
        if not objects_list: # Bỏ qua nếu không có object nào
            continue
            
        # Đường dẫn trực tiếp đến ảnh số 8
        image_path = f"{img_root_path}/{folder_id}/8.jpg"
        img = cv2.imread(image_path)
        
        if img is None:
            print(f">>> LỖI: Không đọc được ảnh {image_path}")
            continue
            
        # Vẽ từng object trong frame
        for obj in objects_list:
            img = draw_object_properties(img, obj)
            
        # Chuyển hệ màu để hiển thị với Matplotlib
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(14, 10))
        plt.imshow(img_rgb)
        plt.title(f"Final Collision Analysis - Video: {folder_id} (Frame {frame_id})", fontsize=16)
        plt.axis('off')
        plt.show()

# Gọi hàm hiển thị
visualize_last_frames_from_final_data(final_data, img_root)

# Bước lọc cuối cùng để chọn ra các thuộc tính 
# Xuất ra file json

In [ ]:
import json
output_json_path = "/kaggle/working/final_results.jsonl"

if os.path.exists(output_json_path):
    os.remove(output_json_path)
    
with open(output_json_path, "a", encoding="utf-8") as f:
    for key, item in final_data.items():
        if len(item) > 0: # item là 1 list các object được lọc
            for obj in item:
                filter_obj = {
                    "folder_id": obj.get('folder_id'),
                    "frame_id": obj.get('frame_id'),
                    "label": obj.get("label"),
                    "probs": obj.get("probs"),
                    'boxs': obj.get('boxs'),
                    "danger_score": obj.get("danger_score"), # LLM dựa vào để ưu tiên
                    "speed": obj.get("speed"),
                    "relative_position": obj.get("relative_postion"),
                    "coming_to_user": obj.get("coming_to_user"),
                    "distance_zone": obj.get("distance_zone")
                }
                # chuyển dict → json string và ghi vào file
                f.write(json.dumps(filter_obj, ensure_ascii=False) + "\n")